# Breast Cancer Fine-Tune — Mistral 7B with QLoRA

**Author:** DiegoDomLarr  
**Goal:** Specialize Mistral-7B-Instruct on breast cancer medical Q&A using QLoRA.

We go step by step:
1. Understand and prepare the data
2. Load the model in 4-bit (QLoRA)
3. Apply LoRA adapter
4. Train
5. Evaluate vs base model
6. Publish to HuggingFace

---

## Session Setup

Run this cell every time you open a new Colab session. It mounts Google Drive (for checkpoint safety) and authenticates with HuggingFace.

> **Before running:** Add your HF write token to Colab Secrets → left sidebar → 🔑 icon → name it `HF_TOKEN`.

In [ ]:
import os, getpass
from google.colab import drive
from huggingface_hub import login

# Mount Google Drive — checkpoints land here so training survives a disconnect
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/FineTuneBreastCancer'
os.makedirs(f'{DRIVE_ROOT}/adapter', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)

# Load HF token — reads from env first (Colab Secrets sets it automatically),
# falls back to manual paste if not found
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF write token: YOUR_HF_TOKEN')

login(token=HF_TOKEN, add_to_git_credential=False)
print('Drive mounted and HuggingFace login done')

## Step 1 — Understand the Data

Before touching a model, we need to understand what we're training on.

We're using **PubMedQA** — a dataset of real biomedical questions derived from PubMed abstracts, with human-verified answers. It lives on HuggingFace and we can load it in one line.

Each example has:
- `question` — the biomedical question
- `context` — the PubMed abstract (paragraphs + MeSH tags)
- `long_answer` — the prose answer we want the model to learn
- `final_decision` — yes / no / maybe

We'll also pull from a second dataset (`lavita/ChatDoctor-HealthCareMagic-100k`) to get more breast cancer examples — because PubMedQA alone only gives us ~30 after filtering.

In [ ]:
# Install the libraries we need for data loading
!pip install datasets -q

In [ ]:
from datasets import load_dataset

# Load PubMedQA — the labeled split has 1000 human-verified examples
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled")

print(pubmedqa)
print("\n--- One example ---")
ex = pubmedqa["train"][0]
print("Question:", ex["question"])
print("Long answer:", ex["long_answer"][:300])
print("Decision:", ex["final_decision"])
print("MeSH tags:", ex["context"]["meshes"])

### Filter for breast cancer

We search across the question, answer, and MeSH tags for breast cancer keywords.

**Problem we discovered:** PubMedQA labeled only gives ~30 breast cancer examples after filtering — too few to fine-tune.

**Solution:** We also load `ChatDoctor-HealthCareMagic-100k`, a large patient Q&A dataset, and filter that for breast cancer too. Combined, we should have 200–400 examples — enough for a meaningful fine-tune.

In [ ]:
# Keywords that indicate breast cancer content
BC_KEYWORDS = [
    "breast cancer", "breast carcinoma", "breast tumour", "breast tumor",
    "BRCA1", "BRCA2", "HER2", "tamoxifen", "mastectomy", "lumpectomy",
    "mammogram", "ductal carcinoma", "lobular carcinoma", "triple negative breast",
    "aromatase inhibitor", "trastuzumab"
]

def is_breast_cancer(text):
    text = text.lower()
    return any(kw.lower() in text for kw in BC_KEYWORDS)

# Filter PubMedQA
def filter_pubmedqa(example):
    combined = example["question"] + " " + example["long_answer"] + " " + " ".join(example["context"]["meshes"])
    return is_breast_cancer(combined)

bc_pubmedqa = pubmedqa["train"].filter(filter_pubmedqa)
print(f"PubMedQA breast cancer examples: {len(bc_pubmedqa)}")

In [ ]:
# Load the second dataset — patient Q&A from real doctor consultations
chatdoctor = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

print(chatdoctor)
print("\n--- One example ---")
print(chatdoctor[0])

In [ ]:
# Filter ChatDoctor for breast cancer
def filter_chatdoctor(example):
    combined = (example.get("input", "") + " " + example.get("output", "") + " " + example.get("instruction", ""))
    return is_breast_cancer(combined)

bc_chatdoctor = chatdoctor.filter(filter_chatdoctor)
print(f"ChatDoctor breast cancer examples: {len(bc_chatdoctor)}")

### Unify into a single format

Both datasets have different field names. We normalize them to a simple structure:
```
{"question": "...", "answer": "..."}
```
Then in the next step we'll convert this into the instruction format Mistral expects.

In [ ]:
import pandas as pd

rows = []

# From PubMedQA: question + long_answer
for ex in bc_pubmedqa:
    if len(ex["long_answer"].strip()) > 50:  # skip very short answers
        rows.append({"question": ex["question"], "answer": ex["long_answer"]})

# From ChatDoctor: instruction/input -> output
for ex in bc_chatdoctor:
    question = (ex.get("instruction", "") + " " + ex.get("input", "")).strip()
    answer = ex.get("output", "").strip()
    if len(question) > 20 and len(answer) > 50:
        rows.append({"question": question, "answer": answer})

df = pd.DataFrame(rows)
print(f"Total breast cancer Q&A pairs: {len(df)}")
print(f"Avg question length: {df['question'].str.len().mean():.0f} chars")
print(f"Avg answer length: {df['answer'].str.len().mean():.0f} chars")
df.head(3)

### Push dataset to HuggingFace

Instead of saving a local JSON that disappears when the Colab session ends, we push the dataset directly to HuggingFace Datasets Hub.

This means any future session (or anyone else) can load it in one line:
```python
load_dataset('DiegoDomLarr/breast-cancer-qa')
```

In [ ]:
from datasets import Dataset

YOUR_HF_TOKEN = Dataset.from_pandas(df)

YOUR_HF_TOKEN.push_to_hub(
    'DiegoDomLarr/breast-cancer-qa',
    token=HF_TOKEN,
    private=False
)

print(f'Pushed {len(YOUR_HF_TOKEN)} examples to DiegoDomLarr/breast-cancer-qa')

### ✅ Step 1 Complete

Your breast cancer Q&A dataset is now live on HuggingFace:
👉 https://huggingface.co/datasets/DiegoDomLarr/breast-cancer-qa

**From any future session, load it with one line:**
```python
from datasets import load_dataset
ds = load_dataset('DiegoDomLarr/breast-cancer-qa')
```

Next: **Step 2 — Understanding QLoRA** (why it works, what the numbers mean)

---

## Step 3 — Load the Model in 4-bit

Before training anything, we load Mistral 7B into Colab using 4-bit quantization (QLoRA).

**What we configure:**
- `load_in_4bit=True` — compress weights from 32-bit floats to 4-bit integers
- `bnb_4bit_quant_type="nf4"` — NormalFloat4, the best 4-bit format for LLM weights
- `bnb_4bit_compute_dtype=float16` — math still runs in float16 for accuracy
- `bnb_4bit_use_double_quant=True` — quantize the quantization constants too (saves ~0.4 GB extra)

**Expected VRAM after loading:** ~4–6 GB (well within the T4's 15 GB limit)

In [ ]:
# Install required libraries — force-upgrade bitsandbytes to meet the minimum version
!pip install -q -U bitsandbytes>=0.46.1 transformers accelerate peft trl
# After this cell runs, go to Runtime → Restart session, then run all cells again

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("4-bit config ready")

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)

print("Model loaded successfully")

In [ ]:
gpu_mem = torch.cuda.memory_allocated() / 1024**3
print(f"GPU memory used: {gpu_mem:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## Step 4 — Apply LoRA

Instead of training all 7B parameters, we inject small trainable matrices (adapters) into the attention layers. The base model stays frozen in 4-bit.

**Key parameters:**
- `r=16` — rank of the adapter matrices. Higher = more capacity, more VRAM.
- `lora_alpha=32` — scales the adapter's influence (typically 2×r)
- `target_modules` — which layers to inject. We target all 4 attention projections in Mistral.
- `lora_dropout=0.05` — small regularization to avoid overfitting on our 1k dataset

**Expected result:** ~1–2% of total params are trainable — everything else stays frozen.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Enable gradients on the frozen quantized model
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 5 — Format Data for Training

Mistral expects a specific instruction format — not raw Q&A pairs. Every training example must look like:

```
<s>[INST] question here [/INST] answer here </s>
```

- `<s>` / `</s>` — sequence boundaries
- `[INST]...[/INST]` — wraps the user instruction
- Everything after `[/INST]` is what the model learns to generate

We load our dataset from HuggingFace and add a `text` column in this format. SFTTrainer will use that column directly.


In [ ]:
from datasets import load_dataset

# Load our breast cancer dataset from HuggingFace
ds = load_dataset("DiegoDomLarr/breast-cancer-qa", split="train")

def format_prompt(example):
    return {
        "text": f"<s>[INST] {example['question']} [/INST] {example['answer']} </s>"
    }

ds = ds.map(format_prompt)

# Verify
print(f"Dataset size: {len(ds)}")
print("\n--- Example formatted entry ---")
print(ds[0]["text"][:400])

## Step 6 — Train

`SFTTrainer` handles the full training loop. We pass it the LoRA-wrapped model, the formatted dataset, and the training config.

**Key hyperparameters:**
- `num_train_epochs=3` — 3 full passes over the dataset
- `per_device_train_batch_size=2` + `gradient_accumulation_steps=4` — effective batch size of 8, safe for T4
- `learning_rate=2e-4` — standard for LoRA fine-tuning
- `max_seq_length=512` — truncates examples to 512 tokens
- `save_steps=50` — checkpoints to Drive every 50 steps in case Colab disconnects

**Expected time:** ~30–40 minutes on a T4 GPU.

In [ ]:
!pip install -q trl

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=f"{DRIVE_ROOT}/adapter",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=sft_config,
)

print("Trainer ready")

In [ ]:
trainer.train()

In [ ]:
  trainer.model.save_pretrained(f"{DRIVE_ROOT}/adapter/final")
  tokenizer.save_pretrained(f"{DRIVE_ROOT}/adapter/final")
  print("Adapter saved to Drive")

In [ ]:
# Switch to inference mode
model.config.use_cache = True
model.eval()

prompt = "<s>[INST] What are the main risk factors for breast cancer? [/INST]"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=== FINE-TUNED MODEL ===")
print(response)

### Compare vs Base Mistral

Now we reload Mistral without the LoRA adapter and ask the same question. This gives us a baseline to compare against.

If the fine-tuned model gives more specific, structured, or medically accurate answers — the training worked. We'll quantify this with ROUGE-L in the next cell.


In [ ]:
# Load base model (no LoRA) for comparison
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)

with torch.no_grad():
    base_inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    base_outputs = base_model.generate(
        **base_inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )

base_response = tokenizer.decode(base_outputs[0], skip_special_tokens=True)
print("=== BASE MISTRAL ===")
print(base_response)

### ROUGE-L Score

ROUGE-L measures the longest common subsequence between two texts — how much the generated answer overlaps with a reference answer.

- Score closer to **1.0** = high overlap (more similar to reference)
- Score closer to **0.0** = low overlap

We use the fine-tuned model's response as the reference, and compare both models against it. A higher ROUGE-L for the fine-tuned model means it learned to produce answers more consistent with the training data style.

In [ ]:
!pip install -q evaluate rouge_score

import evaluate

rouge = evaluate.load("rouge")

# Use fine-tuned response as reference
ft_answer = response.split("[/INST]")[-1].strip()
base_answer = base_response.split("[/INST]")[-1].strip()

results = rouge.compute(
    predictions=[base_answer],
    references=[ft_answer],
    use_stemmer=True,
)

print("=== ROUGE-L COMPARISON ===")
print(f"Fine-tuned response:\n{ft_answer}\n")
print(f"Base Mistral response:\n{base_answer}\n")
print(f"ROUGE-L score (base vs fine-tuned): {results['rougeL']:.4f}")
print("\nA lower score means the base model answers differently than the fine-tuned one.")

In [ ]:
# ▶ CORRER SIEMPRE (si ya entrenaste) — carga el adapter guardado en Drive, evita re-entrenar
from peft import PeftModel

model = PeftModel.from_pretrained(model, f"{DRIVE_ROOT}/adapter/final")
model.eval()
model.config.use_cache = True
print("Fine-tuned model loaded from Drive")

## Step 8 — Publish to HuggingFace Hub

We push the LoRA adapter and tokenizer to HuggingFace so anyone can use it.

The repo will be: `DiegoDomLarr/mistral-7b-breast-cancer-qlora`

To load it in any future session:
```python
from peft import PeftModel
model = PeftModel.from_pretrained(base_model, "DiegoDomLarr/mistral-7b-breast-cancer-qlora")
```

In [ ]:
# Step 8 — Push adapter and tokenizer to HuggingFace Hub
model.push_to_hub(
    "DiegoDomLarr/mistral-7b-breast-cancer-qlora",
    token=HF_TOKEN,
    private=False,
)

tokenizer.push_to_hub(
    "DiegoDomLarr/mistral-7b-breast-cancer-qlora",
    token=HF_TOKEN,
    private=False,
)

print("Adapter and tokenizer pushed to HuggingFace!")
print("https://huggingface.co/DiegoDomLarr/mistral-7b-breast-cancer-qlora")

In [ ]:
from huggingface_hub import HfApi

model_card = """---
language:
- en
license: apache-2.0
tags:
- medical
- breast-cancer
- qlora
- lora
- fine-tuned
- mistral
- peft
- causal-lm
base_model: mistralai/Mistral-7B-Instruct-v0.2
datasets:
- DiegoDomLarr/breast-cancer-qa
pipeline_tag: text-generation
---

# mistral-7b-breast-cancer-qlora

A QLoRA fine-tuned version of [Mistral-7B-Instruct-v0.2](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2) specialized on breast cancer medical question answering.

Trained on a curated dataset of 1,061 breast cancer Q&A pairs assembled from PubMedQA and real patient–doctor consultations. The adapter runs on a single GPU in 4-bit and can be loaded on top of the base Mistral model in minutes.

---

## Why Fine-Tune a Model?

Large language models like Mistral 7B are trained on broad, internet-scale data. That makes them capable generalists — but generalists have limitations when applied to specialized domains.

**Fine-tuning** is the process of continuing the training of a pre-trained model on a smaller, domain-specific dataset. Instead of learning from scratch (which requires massive compute and data), we teach an already-capable model to *speak the language* of a specific field — in this case, breast cancer medicine.

The goal is not to make the model "smarter" in a general sense, but more:
- **Accurate** — using the right clinical terminology and referencing real medical concepts
- **Consistent** — answering breast cancer questions in the structured, informative style of medical Q&A
- **Relevant** — focusing its generation on domain knowledge rather than generic preambles

### Why QLoRA?

Training all 7 billion parameters requires dozens of GB of VRAM and days of compute — out of reach without expensive hardware. **QLoRA (Quantized Low-Rank Adaptation)** makes it accessible with two techniques:

1. **4-bit quantization** — model weights are compressed from 32-bit floats to 4-bit integers, reducing VRAM from ~28 GB to ~5 GB with minimal quality loss.
2. **LoRA adapters** — instead of updating all 7B parameters, small trainable matrices (adapters) are injected into the attention layers. Only ~1–2% of total parameters are trained. The rest stay frozen.

The result: a meaningful domain fine-tune on a single T4 GPU (Google Colab free tier) in approximately 35 minutes.

---

## Model Details

| Property | Value |
|---|---|
| Base model | `mistralai/Mistral-7B-Instruct-v0.2` |
| Fine-tuning method | QLoRA (4-bit quantization + LoRA) |
| LoRA rank (r) | 16 |
| LoRA alpha | 32 |
| Target modules | `q_proj`, `k_proj`, `v_proj`, `o_proj` |
| Training epochs | 3 |
| Effective batch size | 8 (2 per device × 4 gradient accumulation steps) |
| Learning rate | 2e-4 |
| LR scheduler | Cosine |
| Max sequence length | 512 tokens |
| Quantization type | NF4 (NormalFloat4) with double quantization |
| Training hardware | NVIDIA T4 (Google Colab free tier) |
| Training time | ~35 minutes |

---

## Training Data

Trained on **[DiegoDomLarr/breast-cancer-qa](https://huggingface.co/datasets/DiegoDomLarr/breast-cancer-qa)** — 1,061 breast cancer Q&A pairs from two sources:

| Source | Examples | Description |
|---|---|---|
| [PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA) (`pqa_labeled`) | 29 | Human-verified biomedical questions from PubMed abstracts |
| [ChatDoctor-HealthCareMagic-100k](https://huggingface.co/datasets/lavita/ChatDoctor-HealthCareMagic-100k) | 1,032 | Real patient–doctor consultations filtered for breast cancer |
| **Total** | **1,061** | |

**Filter keywords:** `breast cancer`, `breast carcinoma`, `BRCA1`, `BRCA2`, `HER2`, `tamoxifen`, `mastectomy`, `lumpectomy`, `mammogram`, `ductal carcinoma`, `lobular carcinoma`, `triple negative breast`, `aromatase inhibitor`, `trastuzumab`

**Dataset statistics:**
- Average question length: 541 characters
- Average answer length: 621 characters

**Training format** — all examples were converted to Mistral's instruction template:

```
<s>[INST] {question} [/INST] {answer} </s>
```

---

## Evaluation

The fine-tuned model was compared against the base Mistral-7B-Instruct on the prompt:

> *"What are the main risk factors for breast cancer?"*

**Fine-tuned model response:**
> Several factors can increase the risk of developing breast cancer. Here are some of the most common risk factors:
>
> 1. **Gender:** Being female is the greatest risk factor for breast cancer.
> 2. **Age:** The risk of breast cancer increases as women get older. Most breast cancers are diagnosed in women over the age of 50.
> 3. **Genetic Factors:** Certain genetic mutations, such as those in the BRCA1 and BRCA2 genes, can significantly increase the risk. Women with a family history of breast cancer in first-degree relatives are also at higher risk.
> 4. **Lifestyle Factors:** A lack of physical activity, a diet high in saturated fat, being overweight or obese, and smoking all contribute to increased risk.

**Base Mistral response:**
> Breast cancer is the most common cancer among women worldwide. Several risk factors can increase a woman's chance of developing breast cancer.
>
> 1. **Age:** The risk increases as women get older. Most breast cancers are diagnosed after age 50.
> 2. **Genetic factors:** A family history of breast cancer increases the risk. Inherited mutations in BRCA1 and BRCA2 significantly increase risk.
> 3. **Hormonal factors:** Extended exposure to estrogen and progesterone can increase risk. Factors include early menstruation, late menopause, and never having given birth.

**ROUGE-L score (base vs fine-tuned): `0.4509`**

A ROUGE-L of ~0.45 confirms the fine-tuned model generates answers that are meaningfully different from the base — more structured, more patient-oriented, and covering additional factors (gender, lifestyle) not prominently addressed by the base model.

---

## How to Use

### Requirements

```bash
pip install transformers peft bitsandbytes accelerate
```

### Inference

```python
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, "DiegoDomLarr/mistral-7b-breast-cancer-qlora")
model.eval()
model.config.use_cache = True

prompt = "<s>[INST] What are the side effects of tamoxifen? [/INST]"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

---

## Limitations and Intended Use

> **This model is for educational and research purposes only. It is not a medical device and must not be used for clinical diagnosis, treatment decisions, or patient care.**

- Responses may contain inaccuracies or outdated medical information — always verify with a licensed healthcare professional.
- The model was trained on ~1,000 examples, which is small by fine-tuning standards. It may hallucinate or generalize poorly on edge-case questions.
- Coverage is limited to breast cancer topics represented in the training data.
- This model has not been audited, validated, or certified for any medical use.

---

## Tech Stack

| Library | Role |
|---|---|
| `transformers` | Load Mistral-7B and tokenizer |
| `peft` | Apply and load LoRA adapters |
| `bitsandbytes` | 4-bit quantization |
| `trl` | SFTTrainer for supervised fine-tuning |
| `datasets` | Load and process training data |
| `evaluate` | ROUGE-L scoring |
| `huggingface_hub` | Push adapter and dataset to HF Hub |

---

## About This Project

This model was built as an end-to-end learning project covering the full fine-tuning pipeline:

1. Curating and publishing a domain-specific dataset to HF Hub
2. Loading a 7B model in 4-bit on consumer hardware (Colab T4)
3. Applying LoRA adapters with PEFT
4. Training with SFTTrainer (TRL library)
5. Evaluating with ROUGE-L against the base model
6. Publishing the adapter and model card to HuggingFace Hub

**Author:** [DiegoDomLarr](https://huggingface.co/DiegoDomLarr)
**Dataset:** [DiegoDomLarr/breast-cancer-qa](https://huggingface.co/datasets/DiegoDomLarr/breast-cancer-qa)
**Base model:** [mistralai/Mistral-7B-Instruct-v0.2](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2)

---

## License

Apache 2.0 — same as the base model.
"""

api = HfApi()
api.upload_file(
    path_or_fileobj=model_card.encode(),
    path_in_repo="README.md",
    repo_id="DiegoDomLarr/mistral-7b-breast-cancer-qlora",
    token=HF_TOKEN,
)
print("Model card uploaded to HuggingFace!")
print("https://huggingface.co/DiegoDomLarr/mistral-7b-breast-cancer-qlora")

In [ ]:
# Step 8 — Push adapter and tokenizer to HuggingFace Hub
model.push_to_hub(
    "DiegoDomLarr/mistral-7b-breast-cancer-qlora",
    token=HF_TOKEN,
    private=False,
)

tokenizer.push_to_hub(
    "DiegoDomLarr/mistral-7b-breast-cancer-qlora",
    token=HF_TOKEN,
    private=False,
)

print("Adapter and tokenizer pushed to HuggingFace!")
print("https://huggingface.co/DiegoDomLarr/mistral-7b-breast-cancer-qlora")